# YOLO26 training — CARLA Town04 traffic-cam dataset

Trains the vehicle detector for the tailgating pipeline on a Colab GPU (the 8 GB local card can't fit the default settings).

**Before you start:** upload your `dataset/` folder to Google Drive (e.g. `MyDrive/tailgating/dataset`). It needs the `images/{train,val}`, `labels/{train,val}` subfolders and `dataset.yaml`.

Run the cells top to bottom. Each is annotated with what it does and what to check.

## 1. Check the GPU
Confirm Colab gave you a GPU. Runtime → Change runtime type → GPU (T4 on free tier; A100/L4 on Pro). If this errors, you have no GPU allocated.

In [ ]:
!nvidia-smi
# Note the GPU model and memory. T4 = 16 GB (fine). If 'command not found' or no GPU listed,
# switch the runtime to GPU before continuing.

## 2. Install Ultralytics
Pulls the latest ultralytics (includes YOLO26). Pinned import check afterwards.

In [ ]:
!pip -q install -U ultralytics
import ultralytics, torch
ultralytics.checks()
print('torch CUDA available:', torch.cuda.is_available())

## 3. Mount Google Drive
So we can read your uploaded dataset and later save the trained weights back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS to where you put the dataset in Drive:
DRIVE_DATASET = '/content/drive/MyDrive/tailgating/dataset'

import os
assert os.path.isdir(DRIVE_DATASET), f'Not found: {DRIVE_DATASET} - check the path/upload'
print('Found dataset at', DRIVE_DATASET)
print('Contents:', os.listdir(DRIVE_DATASET))

## 4. Copy dataset to local Colab disk
**Important:** train from Colab's local `/content` disk, not directly off Drive. Drive I/O is slow and throttles during training (thousands of small image reads per epoch). Copying once up front makes epochs far faster.

In [ ]:
import shutil, time
LOCAL_DATASET = '/content/dataset'
if os.path.exists(LOCAL_DATASET):
    shutil.rmtree(LOCAL_DATASET)
t0 = time.time()
shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
print(f'Copied to {LOCAL_DATASET} in {time.time()-t0:.0f}s')
for split in ['train', 'val']:
    n_img = len(os.listdir(f'{LOCAL_DATASET}/images/{split}'))
    n_lbl = len(os.listdir(f'{LOCAL_DATASET}/labels/{split}'))
    print(f'  {split}: {n_img} images, {n_lbl} labels')

## 5. Write a Colab-correct dataset.yaml
Your local `dataset.yaml` has an absolute Windows path that won't resolve here. We rewrite it to point at the local Colab copy, keeping the same class order as `site_config.CLASS_NAMES`.

In [ ]:
yaml_text = f'''path: {LOCAL_DATASET}
train: images/train
val: images/val
names:
  0: car
  1: motorcycle
  2: bus
  3: truck
'''
DATA_YAML = '/content/dataset_colab.yaml'
with open(DATA_YAML, 'w') as f:
    f.write(yaml_text)
print(yaml_text)

## 6. Train
Settings tuned for a Colab T4 and this dataset (3k frames, 1920×1080 captures).

- **model**: `yolo26s` is a good speed/accuracy balance on a T4. Bump to `yolo26m` if you have Pro/A100 and want more mAP.
- **imgsz 960**: keeps small distant vehicles resolvable without the full 1920 memory cost.
- **batch -1**: lets Ultralytics auto-pick the largest batch that fits the GPU (safer than hardcoding 16, which may OOM a T4 at imgsz 960).
- **p2 head**: the small-object head helps far-down-range vehicles (60–120 m). Loads from YAML + base backbone (no p2 .pt is released).

First epoch is slow (caching + compile); later epochs speed up. ~100 epochs on 3k frames is a few hours on a T4.

In [ ]:
from ultralytics import YOLO

MODEL   = 'yolo26s'   # n / s / m / l / x
EPOCHS  = 100
IMGSZ   = 960
USE_P2  = True        # small-object head for distant vehicles
PROJECT = '/content/runs/tailgating'
NAME    = 'yolo26_town04'

if USE_P2:
    size = MODEL.replace('yolo26', '')
    net = YOLO(f'yolo26{size}-p2.yaml').load(f'yolo26{size}.pt')
    print(f'P2 head: yolo26{size}-p2.yaml with yolo26{size}.pt backbone')
else:
    net = YOLO(f'{MODEL}.pt')

net.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=-1, device=0,
    project=PROJECT, name=NAME,
    optimizer='MuSGD', close_mosaic=10, augment=True, cache=True, patience=30,
    save_period=10, plots=True,
    lr0=0.001, lrf=0.01, warmup_epochs=3, box=7.5, cls=0.5,
)

## 7. Validate
Report mAP on the val split. mAP50 is the forgiving metric; mAP50-95 is stricter. For a fixed-camera single-scene dataset expect high numbers (the scene is consistent).

In [ ]:
m = net.val(data=DATA_YAML, imgsz=IMGSZ, device=0)
print(f'mAP50: {m.box.map50:.4f}   mAP50-95: {m.box.map:.4f}')
# per-class:
for i, name in enumerate(['car','motorcycle','bus','truck']):
    try:
        print(f'  {name:10s} mAP50-95={m.box.maps[i]:.4f}')
    except Exception:
        pass

## 8. Save weights back to Drive
Colab wipes local disk when the session ends — copy `best.pt` to Drive so you keep it. Download this to your machine and point the perception pipeline at it.

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/tailgating/weights'
os.makedirs(SAVE_DIR, exist_ok=True)
best = f'{PROJECT}/{NAME}/weights/best.pt'
last = f'{PROJECT}/{NAME}/weights/last.pt'
for w in [best, last]:
    if os.path.exists(w):
        dst = f'{SAVE_DIR}/{NAME}_{os.path.basename(w)}'
        shutil.copy(w, dst)
        print('saved', dst)

# also copy the training plots/results for your writeup
results_dir = f'{PROJECT}/{NAME}'
shutil.copytree(results_dir, f'{SAVE_DIR}/{NAME}_run', dirs_exist_ok=True)
print('copied run artifacts (plots, results.csv) to Drive')

## 9. Quick visual sanity check
Run the trained model on a few val images and display predictions. Confirms the weights actually detect vehicles before you take them back to the pipeline.

In [ ]:
import glob, random
from IPython.display import Image, display
val_imgs = glob.glob(f'{LOCAL_DATASET}/images/val/*.png')
sample = random.sample(val_imgs, min(4, len(val_imgs)))
res = net.predict(sample, imgsz=IMGSZ, conf=0.25, save=True, project='/content/pred', name='check')
for p in glob.glob('/content/pred/check/*.png')[:4]:
    display(Image(p))

## Next: back in the pipeline
Download `best.pt` from `MyDrive/tailgating/weights/` to your machine, then:
```python
TrafficCamPerceptionPipeline(camera=cam, yolo_weights='path/to/yolo26_town04_best.pt')
```
That wires your trained detector into Stage 2 → and on to the TTC filtration, reasoning agent, and dashboard.